# Chapter 7 &mdash; Brzozowski's Minimization on `blimp`, End to End

**Concept 12 of the Chapter 7 decomposition:** *A Complete Illustration of Brzozowski's Minimization on `blimp`*

The four steps on a bloated DFA, cross-checked against `min_dfa` at every stage.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Brzozowski-On-Blimp/Concept-Brzozowski-On-Blimp.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Everything from this chapter, applied to one machine.

`blimp` is a deliberately bloated DFA. The four steps &mdash; **rev, det, rev, det** &mdash;
shrink it to the minimal machine, and at each stage you can check what the
intermediate object *is*:

* after step 1: an **NFA** for the reversed language;
* after step 2: a **DFA** for the reversed language, with no equivalent states;
* after step 3: an **NFA** for the original language;
* after step 4: the **minimal DFA**.

Cross-check the final answer against `min_dfa` with `iso_dfa` &mdash; by Myhill&ndash;Nerode
they must agree.

## 2. Definitions

### The bloated machine

In [ ]:
blimp = md2mc('''DFA
I   : 0 -> A
I   : 1 -> B
A   : 0 -> C
A   : 1 -> D
B   : 0 -> D
B   : 1 -> C
C   : 0 -> E
C   : 1 -> F1
D   : 0 -> F1
D   : 1 -> E
E   : 0 | 1 -> E
F1  : 0 | 1 -> F1
''')
print("|Q| =", len(blimp["Q"]), " F =", sorted(blimp["F"]))

### The pipeline, with a report at each stage

In [ ]:
def brz_report(D):
    stages = []
    x = D
    for i, (op, fn) in enumerate([('rev', rev_dfa), ('det', nfa2dfa),
                                  ('rev', rev_dfa), ('det', nfa2dfa)], 1):
        x = fn(x)
        kind = 'NFA' if 'Q0' in x else 'DFA'
        stages.append((i, op, kind, len(x["Q"]), x))
    return stages

## 3. Tests

The four stages, with sizes and machine types.

In [ ]:
stages = brz_report(blimp)
print("%-6s %-6s %-6s %s" % ("step", "op", "kind", "|Q|"))
print("%-6s %-6s %-6s %d" % ("0", "-", "DFA", len(blimp["Q"])))
for i, op, kind, n, _ in stages:
    print("%-6d %-6s %-6s %d" % (i, op, kind, n))
assert stages[0][2] == 'NFA' and stages[1][2] == 'DFA'
assert stages[2][2] == 'NFA' and stages[3][2] == 'DFA'

Stage 2 is a DFA for the **reversed** language &mdash; check it.

In [ ]:
half = stages[1][4]
from itertools import product
strs = [''.join(p) for k in range(10) for p in product('01', repeat=k)]
assert all(accepts_dfa(half, s) == accepts_dfa(blimp, s[::-1]) for s in strs)
print("after two steps the machine accepts exactly the reversed language")

Stage 4 is the minimal DFA for the **original** language.

In [ ]:
final = stages[3][4]
assert all(accepts_dfa(final, s) == accepts_dfa(blimp, s) for s in strs)
print("after four steps the language is back to the original")
print("|Q| : blimp %d -> Brzozowski %d" % (len(blimp["Q"]), len(final["Q"])))

Cross-check against `min_dfa` &mdash; Myhill&ndash;Nerode says they must be isomorphic.

In [ ]:
m = min_dfa(blimp)
print("min_dfa      : %d states" % len(m["Q"]))
print("Brzozowski   : %d states" % len(final["Q"]))
print("iso_dfa      :", iso_dfa(final, m))
assert len(final["Q"]) == len(m["Q"])
assert iso_dfa(final, m)

And `min_dfa_brz` packages the whole pipeline.

In [ ]:
assert iso_dfa(min_dfa_brz(blimp), m)
print("min_dfa_brz agrees with both.  Three independent routes, one minimal machine.")

## 4. Animation

The end of the pipeline: `blimp`, minimized.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa_brz(blimp), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Run the pipeline on the *reverse* of `blimp`. Do you get the same state count?
2. Which stage is the expensive one, and why?
3. Build your own bloated DFA and check both minimizers agree.

In [ ]:
# Your work for the exercises above.